In [8]:
# Import Necessary Libraries
import pandas as pd
from haversine import haversine_vector

In [3]:
# Load the ports data
ports_file_path = "D:\\AIS Project\\datasets\\datasets\\ports.csv"
ports_df = pd.read_csv(ports_file_path, low_memory=False)

In [4]:
# Ensure 'lat' and 'lon' columns are numeric
ports_df['lat'] = pd.to_numeric(ports_df['lat'], errors='coerce')
ports_df['lon'] = pd.to_numeric(ports_df['lon'], errors='coerce')

# Drop rows with invalid latitude or longitude values
ports_df = ports_df.dropna(subset=['lat', 'lon'])
ports_df = ports_df[(ports_df['lat'].between(-90, 90)) & (ports_df['lon'].between(-180, 180))]

In [5]:
# Define the geofence radius (5 nautical miles = 9.26 km)
geofence_radius_km = 9.26

In [6]:
# Load the voyage time and distance differences data
voyage_file_path = "D:\\AIS Project\\datasets\\voyage_time_distance_differences.csv"
voyage_df = pd.read_csv(voyage_file_path, low_memory=False)

In [7]:
# Ensure 'lat' and 'lon' columns in voyage data are numeric
voyage_df['lat'] = pd.to_numeric(voyage_df['lat'], errors='coerce')
voyage_df['lon'] = pd.to_numeric(voyage_df['lon'], errors='coerce')

# Drop rows with invalid latitude or longitude values in voyage data
voyage_df = voyage_df.dropna(subset=['lat', 'lon'])
voyage_df = voyage_df[(voyage_df['lat'].between(-90, 90)) & (voyage_df['lon'].between(-180, 180))]

In [8]:
# Initialize columns to store whether a point is within the geofence and the port name
voyage_df['Within_Port_Geofence'] = False
voyage_df['Port_Name'] = None

# Convert ports data to a list of (lat, lon, name) tuples
ports_coords = list(zip(ports_df['lat'], ports_df['lon'], ports_df['name']))

# Function to check if a point is within the geofence of any port and return the port name
def find_nearest_port(point_lat, point_lon, ports_coords, radius_km):
    # Calculate distances to all ports at once
    distances = haversine_vector([(point_lat, point_lon)] * len(ports_coords), [(lat, lon) for lat, lon, _ in ports_coords])
    # Find the nearest port within the radius
    for i, distance in enumerate(distances):
        if distance <= radius_km:
            return ports_coords[i][2]  # Return the port name
    return None

In [9]:
# Check only the first and last coordinates of each voyage
for (mmsi, voyage_id), group in voyage_df.groupby(['mmsi', 'Voyage_ID']):
    # Get the first and last row of the voyage
    first_row = group.iloc[0]
    last_row = group.iloc[-1]

    # Check if the first coordinate is within any port's geofence
    port_name = find_nearest_port(first_row['lat'], first_row['lon'], ports_coords, geofence_radius_km)
    if port_name:
        voyage_df.loc[first_row.name, 'Within_Port_Geofence'] = True
        voyage_df.loc[first_row.name, 'Port_Name'] = port_name

    # Check if the last coordinate is within any port's geofence
    port_name = find_nearest_port(last_row['lat'], last_row['lon'], ports_coords, geofence_radius_km)
    if port_name:
        voyage_df.loc[last_row.name, 'Within_Port_Geofence'] = True
        voyage_df.loc[last_row.name, 'Port_Name'] = port_name

In [10]:
# Save the updated dataframe to a new file
output_file_path = "D:\\AIS Project\\datasets\\voyage_with_geofence_and_port_name.csv"
voyage_df.to_csv(output_file_path, index=False)

print(f"Geofence data with port names saved to {output_file_path}")

Geofence data with port names saved to D:\AIS Project\datasets\voyage_with_geofence_and_port_name.csv


In [11]:
# Load the voyage data
voyage_file_path = "D:\\AIS Project\\datasets\\voyage_with_geofence_and_port_name.csv"
voyage_df = pd.read_csv(voyage_file_path, low_memory=False)

# Count unique voyages
unique_voyages = voyage_df.groupby(['mmsi', 'Voyage_ID']).ngroups

print(f"Number of unique voyages: {unique_voyages}")

Number of unique voyages: 48936


In [12]:
# Add a new column 'speedKMpH' by multiplying 'speed' by 1.852 (conversion from knots to km/h)
voyage_df['speedKMpH'] = voyage_df['speed'] * 1.852

In [13]:
# Ensure 'timestamp' column is in datetime format
voyage_df['timestamp'] = pd.to_datetime(voyage_df['timestamp'])

# Convert 'Time_Difference' to timedelta format
voyage_df['Time_Difference'] = pd.to_timedelta(voyage_df['Time_Difference'])

# Convert Time_Difference to hours and store in a new column 'Time_Interval'
voyage_df['Time_Interval'] = voyage_df['Time_Difference'].dt.total_seconds() / 3600

# Initialize a new column for weighted average speed
voyage_df['Weighted_Avg_Speed'] = None  # Initialize as None or NaN

# Calculate weighted average speed for each voyage of each mmsi and store it in the first row
for (mmsi, voyage_id), group in voyage_df.groupby(['mmsi', 'Voyage_ID']):
    # Calculate the weighted average speed
    weighted_avg_speed = (group['speedKMpH'] * group['Time_Interval']).sum() / group['Time_Interval'].sum()
    
    # Assign the weighted average speed only to the first row of the voyage
    voyage_df.loc[group.index[0], 'Weighted_Avg_Speed'] = weighted_avg_speed

# Save the updated dataframe to a new file
output_file_path = "D:\\AIS Project\\datasets\\voyage_with_geofence_port_name_weighted_avg_speed.csv"
voyage_df.to_csv(output_file_path, index=False)

print(f"Geofence data with port names, weighted average speed saved to {output_file_path}")

Geofence data with port names, weighted average speed saved to D:\AIS Project\datasets\voyage_with_geofence_port_name_weighted_avg_speed.csv


In [42]:
# Initialize a new column for average course
voyage_df['Avg_Course'] = None  # Initialize as None or NaN

# Calculate average course for each voyage of each mmsi and store it in the first row
for (mmsi, voyage_id), group in voyage_df.groupby(['mmsi', 'Voyage_ID']):
    # Calculate the average course
    avg_course = group['course'].mean()
    
    # Assign the average course only to the first row of the voyage
    voyage_df.loc[group.index[0], 'Avg_Course'] = avg_course

In [43]:
# Initialize a new column for standard deviation of course
voyage_df['Std_Course'] = None  # Initialize as None or NaN

# Calculate standard deviation of course for each voyage of each mmsi and store it in the first row
for (mmsi, voyage_id), group in voyage_df.groupby(['mmsi', 'Voyage_ID']):
    # Calculate the standard deviation of course
    std_course = group['course'].std()
    
    # Assign the standard deviation of course only to the first row of the voyage
    voyage_df.loc[group.index[0], 'Std_Course'] = std_course

In [44]:
# Calculate the change in heading between consecutive data points
voyage_df['heading_change'] = voyage_df.groupby(['mmsi', 'Voyage_ID'])['heading'].diff().abs().fillna(0)

# Aggregate heading change sum at the voyage level
heading_change_sum = voyage_df.groupby(['mmsi', 'Voyage_ID'])['heading_change'].sum().reset_index()

# Initialize a new column for heading change sum
voyage_df['Heading_Change_Sum'] = None  # Initialize as None or NaN

# Assign the heading change sum only to the first row of each voyage
for (mmsi, voyage_id), group in voyage_df.groupby(['mmsi', 'Voyage_ID']):
    # Find the sum for the current voyage
    sum_value = heading_change_sum[
        (heading_change_sum['mmsi'] == mmsi) & 
        (heading_change_sum['Voyage_ID'] == voyage_id)
    ]['heading_change'].values[0]
    
    # Assign the sum to the first row of the voyage
    first_row_index = group.index[0]
    voyage_df.loc[first_row_index, 'Heading_Change_Sum'] = sum_value

# Save the updated dataframe to a new file
output_file_path = "D:\\AIS Project\\datasets\\Transformed_AIS_data.csv"
voyage_df.to_csv(output_file_path, index=False)

print(f"Transformed AIS data saved to {output_file_path}")

Transformed AIS data saved to D:\AIS Project\datasets\Transformed_AIS_data.csv


In [45]:
# Drop duplicate and unnecessary columns
if 'Heading_Change_Sum_x' in voyage_df.columns:
    voyage_df.drop(columns=['Heading_Change_Sum_x'], inplace=True)
if 'Heading_Change_Sum_y' in voyage_df.columns:
    voyage_df.drop(columns=['Heading_Change_Sum_y'], inplace=True)
#if 'Weighted_Speed' in voyage_df.columns:
    #voyage_df.drop(columns=['Weighted_Speed'], inplace=True)
if 'AVGSPDkmph' in voyage_df.columns:
    voyage_df.drop(columns=['AVGSPDkmph'], inplace=True)

# Save the updated dataframe to a new file, replacing the existing file
output_file_path = "D:\\AIS Project\\datasets\\Transformed_AIS_data.csv"
voyage_df.to_csv(output_file_path, index=False)

print(f"Duplicate columns removed and file replaced at {output_file_path}")

Duplicate columns removed and file replaced at D:\AIS Project\datasets\Transformed_AIS_data.csv


In [10]:
# Load the dataset into a DataFrame
file_path = "D:\\AIS Project\\datasets\\Transformed_AIS_data.csv"
voyage_df = pd.read_csv(file_path)

# Calculate weighted average speed for each voyage of each mmsi and store it ONLY in the first row of each voyage
for (mmsi, voyage_id), group in voyage_df.groupby(['mmsi', 'Voyage_ID']):
    # Calculate the weighted average speed using Distance_Difference as weights
    weighted_avg_speed = (group['speedKMpH'] * group['Distance_Difference']).sum() / group['Distance_Difference'].sum()
    
    # Assign the weighted average speed ONLY to the first row of the voyage
    voyage_df.loc[group.index[0], 'AvgSpeedkmph'] = weighted_avg_speed

# Save the updated dataframe to a new file, replacing the existing file
output_file_path = "D:\\AIS Project\\datasets\\Transformed_AIS_data.csv"
voyage_df.to_csv(output_file_path, index=False)

In [12]:
# Initialize a new column for standard deviation of speedKMpH
voyage_df['Std_SpeedKMpH'] = None  # Initialize as None or NaN

# Calculate standard deviation of speedKMpH for each voyage of each mmsi and store it in the first row
for (mmsi, voyage_id), group in voyage_df.groupby(['mmsi', 'Voyage_ID']):
    # Calculate the standard deviation of speedKMpH
    std_speed = group['speedKMpH'].std()
    
    # Assign the standard deviation of speedKMpH only to the first row of the voyage
    voyage_df.loc[group.index[0], 'Std_SpeedKMpH'] = std_speed

# Save the updated dataframe to a new file, replacing the existing file
output_file_path = "D:\\AIS Project\\datasets\\Transformed_AIS_data.csv"
voyage_df.to_csv(output_file_path, index=False)